# Day 4a: How accurate is the gap detector? (hand-labelled check)

You judge every gap the method finds on **15 fresh test images** that were not used to tune it on Day 3,
and count the empty slots it missed. That gives honest **precision** (how many flagged gaps are real)
and **recall** (how many real gaps it finds).

Run on the compute instance (**Python 3.10 - AzureML** kernel). Takes about 30–45 minutes, mostly labelling.

## Setup

In [ ]:
import glob, json, os, sys, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image, ImageOps

SRC = next(os.path.abspath(p) for p in ["../src", "src", "."] if os.path.exists(os.path.join(p, "gaps.py")))
REPO = os.path.dirname(SRC)
sys.path.insert(0, SRC)
from boxes import load_yolo_labels, occupancy_mask, rect_coverage
from gaps import detect_gaps

SKU = os.path.expanduser("~/datasets/SKU-110K")
WEIGHTS = os.path.expanduser("~/runs/full/weights/best.pt")
IMGSZ, CONF = 640, 0.25
N_LABEL, SEED = 15, 7
OUT = os.path.expanduser("~/runs/day4")
os.makedirs(f"{OUT}/label_images", exist_ok=True)

BLUE, ORANGE, VIOLET, INK = "#2a78d6", "#eb6834", "#6250d6", "#52514e"
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb",
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": INK,
    "xtick.color": INK, "ytick.color": INK, "text.color": "#0b0b0b",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#e8e7e3", "grid.linewidth": 0.6,
    "axes.axisbelow": True, "axes.titleweight": "bold", "axes.titlesize": 11,
    "figure.dpi": 110,
})

# Fresh images: exclude the 300 analysed on Day 3 (same seed and size as that notebook)
test_imgs = sorted(glob.glob(f"{SKU}/images/test_*.jpg"))
day3 = set(np.random.default_rng(42).choice(test_imgs, size=min(300, len(test_imgs)), replace=False))
pool = [p for p in test_imgs if p not in day3]
label_set = sorted(np.random.default_rng(SEED).choice(pool, size=min(N_LABEL, len(pool)), replace=False))
names = [os.path.basename(p) for p in label_set]
print(f"{len(pool)} test images not used on Day 3; labelling {len(names)}:")
print(names)

## 1. Detect products and gaps

In [ ]:
# PREDICT
PRED_FILE = f"{OUT}/label_preds.parquet"
if os.path.exists(PRED_FILE):
    pred_df = pd.read_parquet(PRED_FILE)
    print("Loaded saved predictions")
else:
    from ultralytics import YOLO
    model = YOLO(WEIGHTS)
    parts = []
    for path in label_set:  # one path at a time keeps memory flat
        r = model.predict(str(path), imgsz=IMGSZ, conf=CONF, max_det=1000, verbose=False)[0]
        h, w = r.orig_shape
        xyxy = r.boxes.xyxy.cpu().numpy()
        parts.append(pd.DataFrame({"image": os.path.basename(r.path), "w": w, "h": h,
                                   "x1": xyxy[:, 0], "y1": xyxy[:, 1], "x2": xyxy[:, 2], "y2": xyxy[:, 3]}))
    pred_df = pd.concat(parts, ignore_index=True)
    pred_df.to_parquet(PRED_FILE, index=False)
sizes = pred_df.groupby("image")[["w", "h"]].first()

In [ ]:
results, gap_parts = {}, []
for name in names:
    w, h = sizes.loc[name, ["w", "h"]]
    pb = pred_df.loc[pred_df.image == name, ["x1", "y1", "x2", "y2"]].to_numpy(float)
    res = detect_gaps(pb, w, h)
    g = res["gaps"].sort_values(["row", "x1"]).reset_index(drop=True)
    g.insert(0, "gap_id", np.arange(1, len(g) + 1))
    gt = load_yolo_labels(f"{SKU}/labels/{os.path.splitext(name)[0]}.txt", w, h)
    mask = occupancy_mask(gt, w, h)
    g["label_cover"] = [rect_coverage(mask, r) for r in g[["x1", "y1", "x2", "y2"]].to_numpy()]
    results[name] = (pb, res, g)
    gap_parts.append(g.assign(image=name, check_layout=res["check_layout"], gap_score=res["score"]))

gaps_df = pd.concat(gap_parts, ignore_index=True)
summary = gaps_df.groupby("image").agg(gaps=("gap_id", "size")).reindex(names, fill_value=0)
summary["check_layout"] = [results[n][1]["check_layout"] for n in names]
summary["gap_score"] = [round(results[n][1]["score"], 3) for n in names]
print(f"{len(gaps_df)} gaps to judge across {len(names)} images")
summary

## 2. Label the gaps

Each image is drawn large with its gaps **numbered**. Open the saved PNGs in `~/runs/day4/label_images/`
to zoom in if you need to.

**Labelling guide.** Decide from the photo alone, not from the numbers:

| Call | Means |
|---|---|
| **Real** | An empty facing where a product should stand: bare shelf or back panel visible, often a price tag with nothing above it, and at least roughly one product wide. |
| **Not real** | Normal spacing between product groups, space above shorter products, a door frame or upright, peg-hook spacing, a product that is there but dark, hidden or partly cut off, signs or promo material. |
| **Unsure** | You genuinely can't tell (blurry, too far away). Reported separately, not counted. |
| **Missed** | A clear empty facing at least one product wide **between** products that has **no** orange box. Ignore the ends of shelves: the method deliberately skips them. |

Blue = detected products, orange numbered boxes = gaps to judge, dotted = fixture gaps already removed (don't judge these).

In [ ]:
def draw_for_labelling(name, ax):
    pb, res, g = results[name]
    w, h = sizes.loc[name, ["w", "h"]]
    ax.imshow(ImageOps.exif_transpose(Image.open(f"{SKU}/images/{name}")))
    for x1, y1, x2, y2 in pb:
        ax.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor=BLUE, linewidth=0.4))
    for f in res["structural_gaps"].itertuples():
        ax.add_patch(Rectangle((f.x1, f.y1), f.x2 - f.x1, f.y2 - f.y1, fill=False,
                               edgecolor=INK, linewidth=0.8, linestyle=":"))
    for r in g.itertuples():
        ax.add_patch(Rectangle((r.x1, r.y1), r.x2 - r.x1, r.y2 - r.y1, fill=False,
                               edgecolor=ORANGE, linewidth=2))
        ax.text((r.x1 + r.x2) / 2, r.y1, str(r.gap_id), ha="center", va="bottom", fontsize=11,
                fontweight="bold", color="white",
                bbox=dict(boxstyle="round,pad=0.15", facecolor=ORANGE, edgecolor="none"))
    ax.set_xlim(0, w); ax.set_ylim(h, 0); ax.axis("off"); ax.grid(False)
    flag = "  |  CHECK LAYOUT" if res["check_layout"] else ""
    ax.set_title(f"{name}: {len(g)} gaps to judge{flag}", fontsize=11)


for name in names:
    fig, ax = plt.subplots(figsize=(10, 12))
    draw_for_labelling(name, ax)
    plt.tight_layout()
    plt.savefig(f"{OUT}/label_images/{os.path.splitext(name)[0]}.png", dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)

### Fill in your labels
Run the next cell: it prints a template with every image. **Copy it into the cell after it**, then for each image
list the gap numbers you judge real and unsure (everything else counts as not real) and how many empty slots it missed.

In [ ]:
print("LABELS = {")
for name in names:
    n = int(summary.loc[name, "gaps"])
    print(f'    "{name}": {{"real": [], "unsure": [], "missed": None, "note": ""}},  # {n} gaps')
print("}")

In [ ]:
# Paste the template here and fill it in, e.g.
# "test_123.jpg": {"real": [1, 4], "unsure": [2], "missed": 1, "note": "pharmacy shelf"},
LABELS = {
}

## 3. Results

In [ ]:
problems = [n for n in names if n not in LABELS or LABELS[n].get("missed") is None]
for n, lab in LABELS.items():
    ids = set(lab["real"]) | set(lab["unsure"])
    n_gaps = int(summary.loc[n, "gaps"]) if n in summary.index else 0
    if any(i < 1 or i > n_gaps for i in ids) or set(lab["real"]) & set(lab["unsure"]):
        problems.append(n)
assert not problems, f"Fix the labels for: {sorted(set(problems))}"


def wilson(k, n, z=1.96):
    """95% Wilson interval for a proportion; reliable for small samples."""
    if n == 0:
        return (np.nan, np.nan, np.nan)
    p = k / n
    centre = (p + z**2 / (2 * n)) / (1 + z**2 / n)
    half = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / (1 + z**2 / n)
    return p, centre - half, centre + half


def call(row):
    lab = LABELS[row.image]
    return "real" if row.gap_id in lab["real"] else "unsure" if row.gap_id in lab["unsure"] else "not real"


gaps_df["call"] = gaps_df.apply(call, axis=1) if len(gaps_df) else []
per_img = pd.DataFrame({
    "gaps": summary["gaps"],
    "real": [len(LABELS[n]["real"]) for n in names],
    "unsure": [len(LABELS[n]["unsure"]) for n in names],
    "missed": [LABELS[n]["missed"] for n in names],
    "check_layout": summary["check_layout"],
    "gap_score": summary["gap_score"],
}, index=names)
per_img["not_real"] = per_img.gaps - per_img.real - per_img.unsure


def report(df, label):
    tp, fp, fn = df.real.sum(), df.not_real.sum(), df.missed.sum()
    p, plo, phi = wilson(tp, tp + fp)
    r, rlo, rhi = wilson(tp, tp + fn)
    print(f"{label}: {len(df)} images | real {tp}, not real {fp}, unsure {df.unsure.sum()}, missed {fn}")
    print(f"   precision {p:.0%} (95% CI {plo:.0%}-{phi:.0%}) | recall {r:.0%} (95% CI {rlo:.0%}-{rhi:.0%})")
    return {"images": len(df), "real": int(tp), "not_real": int(fp), "missed": int(fn),
            "precision": p, "precision_lo": plo, "precision_hi": phi,
            "recall": r, "recall_lo": rlo, "recall_hi": rhi}


metrics = {"all": report(per_img, "All images")}
normal = per_img[~per_img.check_layout]
if len(normal) < len(per_img):
    metrics["not_flagged"] = report(normal, "Images not flagged 'check layout'")
per_img

### Does the Day 3 label check mean anything?
On Day 3, a gap was "empty in the labels too" when the dataset's boxes didn't cover it.
If that check is useful, those gaps should be real far more often than gaps the labels cover.

In [ ]:
judged = gaps_df[gaps_df.call != "unsure"].copy()
judged["labels_empty"] = np.where(judged.label_cover <= 0.5, "Empty in labels", "Labels show a product")
check = judged.groupby("labels_empty")["call"].agg(gaps="size", real=lambda s: (s == "real").sum())
check["share_real"] = (check.real / check.gaps).round(2)
check

### Does the gap score rank shelves sensibly?
Images with more real gaps should have higher gap scores. With 15 images this is only a rough check.

In [ ]:
from scipy.stats import spearmanr

rho, pval = spearmanr(per_img.gap_score, per_img.real + per_img.missed)
print(f"Spearman correlation between gap score and real empty slots: {rho:.2f} (p = {pval:.2f}, n = {len(per_img)})")

fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(per_img.gap_score, per_img.real + per_img.missed, s=40, color=BLUE,
           edgecolor="#fcfcfb", linewidth=1.5, zorder=3)
ax.set(xlabel="Gap score from the method", ylabel="Real empty slots (your labels)",
       title="Higher gap scores should mean more empty slots")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
plt.tight_layout(); plt.savefig(f"{OUT}/score_vs_real.png", bbox_inches="tight"); plt.show()

## 4. Save labels and results

In [ ]:
os.makedirs(f"{REPO}/labels", exist_ok=True)
with open(f"{REPO}/labels/gap_labels_day4.json", "w") as f:  # small and valuable: commit it
    json.dump({"images": names, "seed": SEED, "weights": WEIGHTS, "labels": LABELS}, f, indent=2)
with open(f"{OUT}/gap_check_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2, default=float)
gaps_df.to_parquet(f"{OUT}/labelled_gaps.parquet", index=False)
per_img.to_csv(f"{OUT}/labelled_images.csv")
print("Saved labels to", f"{REPO}/labels/gap_labels_day4.json")

## Notes for LOG.md
- Precision and recall with their 95% intervals (15 images is a small sample, so the intervals are wide; report them).
- Whether the Day 3 label check separates real from false gaps.
- The most common reasons for "not real" and for misses (from your notes).